## Define functions

In [89]:
"""Helper functions for PACE Hackweek Validation Tutorial.

Authors:
    James Allen and Anna Windle
"""

import datetime
import os
import re
from pathlib import Path

import earthaccess
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.style as style
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.ticker import FuncFormatter
from scipy import odr, stats
import requests

# AERONET-OC Download Constants
# Valid AERONET-OC site list
DF_AERONET_SITES = pd.read_csv(
    "https://aeronet.gsfc.nasa.gov/aeronet_locations_v3.txt",
    delimiter=",",
    skiprows=1
    )
AERONET_SITES = list(DF_AERONET_SITES["Site_Name"].sort_values())
OCEAN_SITES = [
    "AAOT",
    "Abu_Al_Bukhoosh",
    "ARIAKE_TOWER",
    "Bahia_Blanca",
    "Banana_River",
    "Blyth_NOAH",
    "Casablanca_Platform",
    "Chesapeake_Bay",
    "COVE_SEAPRISM",
    "Galata_Platform",
    "Gloria",
    "GOT_Seaprism",
    "Grizzly_Bay",
    "Gustav_Dalen_Tower",
    "Helsinki_Lighthouse",
    "Ieodo_Station",
    "Irbe_Lighthouse",
    "Kemigawa_Offshore",
    "Lake_Erie",
    "Lake_Okeechobee",
    "Lake_Okeechobee_N",
    "LISCO",
    "Lucinda",
    "MVCO",
    "Palgrunden",
    "PLOCAN_Tower",
    "RdP-EsNM",
    "Sacramento_River",
    "San_Marco_Platform",
    "Section-7_Platform",
    "Socheongcho",
    "South_Greenbay",
    "Thornton_C-power",
    "USC_SEAPRISM",
    "Venise",
    "WaveCIS_Site_CSI_6",
    "Zeebrugge-MOW1",
]

# Get subset of AERONET columns to make it a bit more manageable (also rename)
AOC_KEEP_COLS = [
    "AERONET_Site",
    "field_datetime",
    "Site_Latitude(Degrees)",
    "Site_Longitude(Degrees)",
    "Solar_Zenith_Angle[400nm]",
]
COLUMN_RENAME = {
    "Site_Latitude(Degrees)": "field_latitude",
    "Site_Longitude(Degrees)": "field_longitude",
    "AERONET_Site": "field_site",
    "Solar_Zenith_Angle[400nm]": "field_solar_zenith",
}

# Bland-Altman/Scatterplot Constants
# Plot colors, font sizes
COLOR_PALETTE = sns.color_palette("colorblind")
COLOR_SCATTER = COLOR_PALETTE[0]
COLOR_LINE = "black"  # Was "black"
COLOR_LOA = COLOR_PALETTE[2]  # Was "green"
COLOR_FITLINE = COLOR_PALETTE[1]  # Was "magenta"
SIZE_TITLE = 24
SIZE_AXLABEL = 20
SIZE_TEXTLABEL = 14
SHOW_LEGEND = False

# Update some defaults
plt.rcParams.update({"figure.dpi": 300})
sns.set_style("ticks", rc={"figure.dpi": 300})
sns.set_context("notebook", font_scale=1.45)

# Satellite Matchup Constants
# Short names for earthaccess lookup
SAT_LOOKUP = {
    "PACE_AOP": "PACE_OCI_L2_AOP",
    "PACE_IOP": "PACE_OCI_L2_IOP",
    "PACE_BGC": "PACE_OCI_L2_BGC",
    "PACE_PAR": "PACE_OCI_L2_PAR",
    "AQUA": "MODISA_L2_OC",
    "TERRA": "MODIST_L2_OC",
    "NOAA-20": "VIIRSJ1_L2_OC",
    "NOAA-21": "VIIRSJ2_L2_OC",
    "SUOMI-NPP": "VIIRSN_L2_OC",
}

# List l2 flags, then build them into a dict
l2_flags_list = [
    "ATMFAIL",
    "LAND",
    "PRODWARN",
    "HIGLINT",
    "HILT",
    "HISATZEN",
    "COASTZ",
    "SPARE",
    "STRAYLIGHT",
    "CLDICE",
    "COCCOLITH",
    "TURBIDW",
    "HISOLZEN",
    "SPARE",
    "LOWLW",
    "CHLFAIL",
    "NAVWARN",
    "ABSAER",
    "SPARE",
    "MAXAERITER",
    "MODGLINT",
    "CHLWARN",
    "ATMWARN",
    "SPARE",
    "SEAICE",
    "NAVFAIL",
    "FILTER",
    "SPARE",
    "BOWTIEDEL",
    "HIPOL",
    "PRODFAIL",
    "SPARE",
]
L2_FLAGS = {flag: 1 << idx for idx, flag in enumerate(l2_flags_list)}

# Bailey and Werdell 2006 exclusion criteria
EXCLUSION_FLAGS = [
    "LAND",
    "HIGLINT",
    "HILT",
    "STRAYLIGHT",
    "CLDICE",
    "ATMFAIL",
    "LOWLW",
    "FILTER",
    "NAVFAIL",
    "NAVWARN",
]

# OCSSW Dataroot folder for tables
# OCDATAROOT = Path(os.environ.get("OCSSWROOT")).resolve() / "share"
OCDATAROOT = "/private/tmp/ocssw/share/"
OCI_SENSOR_FILE = OCDATAROOT + "oci/msl12_sensor_info.dat"

##---------------------------------------------------------------------------##
#                              General Utilities                              #
##---------------------------------------------------------------------------##


def get_f0(wavelengths=None, window_size=10):
    """Load the OCI sensor file and return F0.

    Defaults to returning the full table. Input obs_time to correct for the
    Earth-Sun distance.

    Parameters
    ----------
    sensor_file : str or pathlib.Path
        Path to the OCI satellite sensor file containing wavelengths and F0.
    wavelengths : array-like, optional
        Wavelengths at which to compute the average irradiance.
        If None, returns the full wavelength and irradiance table.
    window_size : int, optional
        Bandpass filter size for mean filtering to selected wavelengths, in nm.

    Returns
    -------
    tuple of np.ndarray
        A tuple containing:
        - f0_spectra : np.ndarray
            The extraterrestrial solar irradiance, in uW/cm^2/nm.
        - f0_wave : np.ndarray
            The corresponding wavelengths, in nm.

    """
    with open(OCI_SENSOR_FILE, "r") as file_in:
        for line in file_in:
            if "Nbands" in line:
                (key, nbands) = line.split("=")
                break

    wl = np.zeros(int(nbands), dtype=float)
    f0 = np.zeros(int(nbands), dtype=float)
    with open(OCI_SENSOR_FILE, "r") as file_in:
        for line in file_in:
            if "=" in line:
                (key, value) = line.split("=")
                if "Lambda" in key:
                    idx = re.findall(r"\d+", key)
                    wvlidx = int(idx[0]) - 1
                    wl[wvlidx] = float(value)
                if "F0" in key:
                    idx = re.findall(r"\d+", key)
                    wvlidx = int(idx[1]) - 1
                    f0[wvlidx] = float(value)

    if wavelengths is not None:
        f0_wave = np.array(wavelengths)
        f0_spectra = bandpass_avg(f0, wl, window_size, f0_wave)
    else:
        f0_wave = wl
        f0_spectra = f0

    return f0_spectra, f0_wave


def bandpass_avg(
        data,
        input_wavelengths,
        window_size=10,
        target_wavelengths=None
        ):
    """Apply a band-pass filter to the data.

    Parameters
    ----------
    data : np.ndarray
        1D or 2D array containing the spectral data (samples x wavelengths).
        If 1D, it's assumed to be a single sample.
    input_wavelengths : np.ndarray
        1D array of wavelength values corresponding to the columns of data.
    window_size : int, optional
        Size of the window to use for averaging. Default is 10 nm.
    target_wavelengths : np.ndarray, optional
        1D array of target wavelengths for filtered values.
        If None, the input wavelengths are used.

    Returns
    -------
    np.ndarray
        1D or 2D array containing the band-pass 
        data.

    """
    data = np.atleast_2d(data)
    half_window = window_size / 2
    num_samples, num_input_wavelengths = data.shape
    if target_wavelengths is None:
        target_wavelengths = input_wavelengths

    filtered_data = np.empty((num_samples, len(target_wavelengths))) * np.nan

    for idx, target_wl in enumerate(target_wavelengths):
        start = target_wl - half_window
        end = target_wl + half_window
        cols_in_range = np.where(
            (input_wavelengths >= start) & (input_wavelengths <= end)
        )[0]
        if cols_in_range.size > 0:
            filtered_data[:, idx] = np.nanmean(data[:, cols_in_range], axis=1)

    return filtered_data if num_samples > 1 else filtered_data.flatten()


def get_column_prods(df, type_prefix):
    """Process a dataframe to create a dictionary of data products.

    Parameters
    ----------
    df : pandas DataFrame
        Extracted dataframes from read_extract_file
    type_prefix : str
        Prefix to identify the product columns, e.g. "aoc"

    Returns
    -------
    data_dict
        dictionary mapping data product with their wavelengths and columns.

    """
    data_dict = {}
    pattern = rf"{type_prefix}_(\w+?)(\d*\.?\d+)?$"

    for col in df.columns:
        match = re.match(pattern, col)
        if match:
            product = match.group(1)
            wavelength = match.group(2) if match.group(2) else None
            if product not in data_dict:
                data_dict[product] = {"wavelengths": [], "columns": []}
            data_dict[product]["columns"].append(col)
            if wavelength:
                if "." in wavelength:
                    data_dict[product]["wavelengths"].append(float(wavelength))
                else:
                    data_dict[product]["wavelengths"].append(int(wavelength))
    return data_dict


def read_sb(filename_sb):
    """Read SeaBASS file and returns just the data.

    Input
    -----
    filename_sb : str
        path to seabass file

    Output
    ------
    data : pandas dataframe object
        seabass data from file
    """
    with open(filename_sb, "r") as file:
        lines = [line.rstrip() for line in file]

    # Parse headers, get index where they end
    idx_endheader = [index for index, value in enumerate(lines)
                     if value == "/end_header"]
    header_lines = lines[1:idx_endheader[0]]
    headers = dict()
    comments = []
    for header_line in header_lines:
        if header_line.startswith("!"):
            # Separate out the comments
            comments.append(header_line)
        else:
            # Split the header and add to the dictionary
            key, value = header_line.split("=", 1)
            headers[key[1:]] = value  # Remove leading "/" from key

    # Pull data into pandas dataframe
    data = pd.read_csv(filename_sb,
                       skiprows=idx_endheader[0]+1,
                       names=headers["fields"].split(","),
                       na_values=headers["missing"])

    # Index by datetime
    get_sb_datetime(data)

    return data


def get_sb_datetime(df):
    """Parse datetime from different combinations of dates and times."""
    if all(col in df.columns for col in ["year", "month", "day",
                                         "hour", "minute", "second"]):
        df["datetime"] = pd.to_datetime(df[["year", "month", "day",
                                            "hour", "minute", "second"]])
    elif all(col in df.columns for col in ["year", "month", "day", "time"]):
        df["datetime"] = pd.to_datetime(
            df["year"].astype(str) + df["month"].astype(str).str.zfill(2)
            + df["day"].astype(str).str.zfill(2) + ' ' + df["time"])
    elif all(col in df.columns for col in ["date", "time"]):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + ' ' + df["time"])
    elif all(col in df.columns for col in ["year", "month", "day"]):
        df["datetime"] = pd.to_datetime(df[["year", "month", "day"]])
    elif all(col in df.columns for col in ["date", "hour",
                                           "minute", "second"]):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + ' ' + df["hour"].astype(str).str.zfill(2)
            + ':' + df["minute"].astype(str).str.zfill(2) + ':'
            + df["second"].astype(str).str.zfill(2))
    else:
        print("Unrecognized date/time format in DataFrame columns."
              "\nMay be a profile, but doublecheck.")
        return

    # Reindex the dataframe with the new datetime
    df.set_index("datetime", inplace=True)


##---------------------------------------------------------------------------##
#                             Satellite Utilities                             #
##---------------------------------------------------------------------------##


def parse_quality_flags(flag_value):
    """Parse bitwise flag into a list of flag names.

    Parameters
    ----------
    flag_value : int
        The integer representing the combined bitwise quality flags.

    Returns
    -------
    list of str
        List of flag names that are set in the flag_value.

    """
    return [
        flag_name for flag_name, value in L2_FLAGS.items()
        if (flag_value & value) != 0
    ]


def get_fivebyfive_Rrs(file, latitude, longitude, par, wavelengths, rrs_wavelengths):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups
    wavelengths ; numpy array
        Desired Rrs wavelengths to match
    rrs_wavelengths ; numpy array
        Rrs wavelengths (from wavelength_3d for OCI)

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: This is hard-coded to Rrs from an L2 AOP file.
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        rrs_data = (
            ds_data[par].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        
        # Select only the desired wavelengths
        rrs_indices = [np.where(rrs_wavelengths == wl)[0][0] for wl in wavelengths]
        rrs_data = rrs_data[:, :, rrs_indices]

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any() and np.isnan(rrs_data[valid_mask]).all() == False:
        rrs_valid = rrs_data[valid_mask]
        rrs_std_initial = np.std(rrs_valid, axis=0)
        rrs_mean_initial = np.mean(rrs_valid, axis=0)

        # Exclude spectra > 1.5 stdevs away
        std_mask = np.all(
            np.abs(rrs_valid - rrs_mean_initial) <= 1.5 * rrs_std_initial,
            axis=1
        )
        rrs_std = np.std(rrs_valid[std_mask], axis=0)
        rrs_mean = np.mean(rrs_valid[std_mask], axis=0).flatten()
    else:
        valid_mask[:] = False
        rrs_mean = np.nan * np.empty_like(wavelengths)

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
    }

    # Add mean spectra to the row dictionary
    for wavelength, mean_value in zip(wavelengths, rrs_mean):
        key = f"sat_{par.lower()}{int(wavelength)}"
        row[key] = mean_value

    return row


def get_fivebyfive_PAR(file, latitude, longitude):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: Using "par_day_planar_above" product for PAR data, which I assume is what standard OC PAR is...
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        par_data = (
            ds_data["par_day_planar_above"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any():
        par_valid = par_data[valid_mask]
        par_mean = np.mean(par_valid, axis=0)
    else:
        par_mean = np.nan

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
        "sat_par": par_mean
    }

    return row


def get_sat_matchups(
    start_date,
    end_date,
    latitude,
    longitude,
    wavelengths="all",
    par="Rrs",
    sat="PACE_AOP",
    selected_dates=None
):
    """Make satellite timeseries of matchups from single station.

    Caution: If the date or coordinates aren't formatted correctly, it might
    pull a huge granule list and take forever to run. If it takes more than 45
    seconds to print the number of granules, just kill the process.

    Uses the earthaccess package. Defaults to the PACE OCI L2 IOP datasets,
    but other satellites can be used if they have a corresponding short_name
    in the SAT_LOOKUP dictionary.

    Workflow:
        1. Get list of matchup granules
        2. Loop through each file and:
            2a. Find closest pixel to station, extract 5x5 pixel box
            2b. Exclude pixels based on l2_flags
            2c. Filtered mean to get single spectra
            2d. Compute statistics and save data row
        3. Organize output pandas dataframe

    Parameters
    ----------
    start_date : datetime or str
        Beginning of Aeronet data to run.
    end_date : datetime or str, optional
        End of Aeronet data to run.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups
    sat : str
        Name of satellite to search. Must be in SAT_LOOKUP dict constant.
    selected_dates : list of str, optional
        If given, only pull granules if the dates are in this list

    Returns
    -------
    pandas DataFrame object
        Flattened table of all satellite granule matchups.

    """
    # Look up short name from constants
    if sat not in SAT_LOOKUP.keys():
        raise ValueError(
            f"{sat} is not in the lookup dictionary. Available "
            f"sats are: {', '.join(SAT_LOOKUP)}"
        )
    short_name = SAT_LOOKUP[sat]

    # Format search parameters
    time_bounds = (f"{start_date}T00:00:00", f"{end_date}T23:59:59")

    # Run Earthaccess data search
    results = earthaccess.search_data(
        point=(longitude, latitude),
        temporal=time_bounds,
        short_name=short_name
    )
    if selected_dates is not None:
        filtered_results = [
            result
            for result in results
            if result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10]
            in selected_dates
        ]
        print(f"    Filtered to {len(filtered_results)} Granules.")
        files = earthaccess.open(filtered_results, pqdm_kwargs={"disable": True})
    else:
        files = earthaccess.open(results)

    # Get 5x5 pixel data
    if sat == "PACE_AOP" or sat == "PACE_IOP":
        if len(files) > 0:
            # Pull out all available Rrs wavelengths
            with xr.open_dataset(files[0], group="sensor_band_parameters") as ds_bands:
                rrs_wavelengths = ds_bands["wavelength_3d"].values
            
            # Select wavelengths or interest
            if wavelengths is None or wavelengths == "all":
                wavelengths = rrs_wavelengths
            else:
                # Get nearest wavelengths to desired input wavelengths
                nearest_wavelengths = []
                for target_wl in wavelengths:
                    nearest_wl = rrs_wavelengths[np.abs(rrs_wavelengths - target_wl).argmin()]
                    nearest_wavelengths.append(nearest_wl)
                wavelengths = np.array(nearest_wavelengths)
            
            # Loop through files and process
            sat_rows = []
            for idx, file in enumerate(files):
                granule_date = pd.to_datetime(
                    file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
                )
                print(f"    Running Granule: {granule_date}")
                row = get_fivebyfive_Rrs(file, latitude, longitude, par, wavelengths, rrs_wavelengths)
                sat_rows.append(row)
        else:
            sat_rows = []
            
    elif sat == "PACE_PAR":
        # Loop through files and process
        sat_rows = []
        for idx, file in enumerate(files):
            granule_date = pd.to_datetime(
                file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
            )
            print(f"    Running Granule: {granule_date}")
            row = get_fivebyfive_PAR(file, latitude, longitude)
            sat_rows.append(row)
        
    return pd.DataFrame(sat_rows)
    

In [85]:
# TESTING CODE FOR SATELLITE MATCHUPS
i = 5
date = all_dates[i]
lat = all_lat[i]
lon = all_lon[i]
df_Rrs = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        par="Rrs",
        sat="PACE_AOP",
        selected_dates=[date],
        )

print(df_Rrs)

    Filtered to 2 Granules.
    Running Granule: 2025-01-25 07:37:39+00:00
    Running Granule: 2025-01-25 09:15:57+00:00
               sat_datetime  sat_latitude  sat_longitude  sat_pixel_valid  \
0 2025-01-25 07:37:39+00:00    -50.405060      70.967964                0   
1 2025-01-25 09:15:57+00:00    -50.403625      70.943604                9   

   sat_rrs413  sat_rrs442  sat_rrs490  sat_rrs555  sat_rrs670  
0         NaN         NaN         NaN         NaN         NaN  
1    0.005631    0.004723    0.004659    0.003928    0.001061  


## Create profile list for satellite matchups

In [90]:
fileDir = "/Volumes/Data/Work/SEALTAGS/Prydz/ct182/v5_20251106/PROCESSED/NC/"

df_profiles = pd.DataFrame(columns=["SEALTAG_NC_FILE", "TAG_ID", "PROFILE_NUM", "DATE", "LATITUDE", "LONGITUDE","ISDAYTIME","PROCESSED"])

# Loop through profile information excel files in directory
for file in sorted(os.listdir(fileDir)):
    if file.endswith(".xlsx") and not file.startswith("~$"):
        profile_info_file = os.path.join(fileDir, file)
        df_General  = pd.read_excel(profile_info_file, sheet_name="General")
        
        # If noData==TRUE in df_Light, skip file
        df_Light = pd.read_excel(profile_info_file, sheet_name="LIGHT")
        if df_Light['noData'].all():
            continue
        
        # Create table with 3 colunns: SEALTAG_NC_FILE, TAG_ID, PROFILE_ID
        tag_id = df_General['PlatformID']
        profile_num = df_General['Profile']
        file_nc = profile_info_file.replace('_ProfileInfo.xlsx', '_PROCESSED.nc')
        file_nc = [file_nc] * len(tag_id)
        lat = df_General['Lat']
        lon = df_General['Lon']
        date = df_General['Date']
        isDaytime = df_General['Daytime']
        
         # Append each profile to dataframe
        for i in range(len(tag_id)):
            new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [file_nc[i]],
                "TAG_ID": [tag_id.values[i]],
                "PROFILE_NUM": [profile_num.values[i]],
                "DATE": [date.values[i]],
                "LATITUDE": [lat.values[i]],
                "LONGITUDE": [lon.values[i]],
                "ISDAYTIME": [isDaytime.values[i]],
                "PROCESSED": [False]
            })
            df_profiles = pd.concat([df_profiles, new_row], ignore_index=True)
            
# If satellite_matchups.csv exists, find the latest processed profile and mark as PROCESSED
sat_matchups_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/satellite_matchups.csv"
if os.path.exists(sat_matchups_file):
    df_sat_matchups = pd.read_csv(sat_matchups_file, delimiter=",", skiprows=0)
    # Get latest processed profile and tag ID
    latest_processed_profilenum = df_sat_matchups["PROFILE_NUM"][len(df_sat_matchups)-1]
    latest_processed_tagID = df_sat_matchups["TAG_ID"][len(df_sat_matchups)-1]
    
    # Find row index in df_profiles with matching TAG_ID and PROFILE_NUM
    rowidx = df_profiles[
        (df_profiles["TAG_ID"] == latest_processed_tagID) &
        (df_profiles["PROFILE_NUM"] == latest_processed_profilenum)
    ].index[0]
    
    # Set PROCESSED=True for all rows prior to and including this row
    df_profiles.loc[0:rowidx, ["PROCESSED"]] = True

# Save dataframe to csv
profile_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/profiles_to_match.csv"
df_profiles.to_csv(profile_csv_file, index=False)

all_df = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
all_df["DATE"] = pd.to_datetime(all_df["DATE"])

all_ids = all_df["TAG_ID"].values
all_profilnum = all_df["PROFILE_NUM"].values
all_lat = all_df["LATITUDE"].values
all_lon = all_df["LONGITUDE"].values
all_dates = all_df["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = all_df["SEALTAG_NC_FILE"].values
all_isdaytime = all_df["ISDAYTIME"].values
all_processed = all_df["PROCESSED"].values

/var/folders/t1/9ygzcs0j2ms2w3wgpmhfdhn1857szx/T/ipykernel_23481/1772371060.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_profiles = pd.concat([df_profiles, new_row], ignore_index=True)


## Get PACE OCI matchups

In [91]:
from colorama import Fore, Back, Style

# Read in existing matchup CSV if it exists, otherwise create new one
matchup_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/satellite_matchups.csv"
if not os.path.exists(matchup_csv_file):
    # Create empty dataframe with columns
    df_matchups = pd.DataFrame(columns=[
        "SEALTAG_NC_FILE",
        "TAG_ID",
        "PROFILE_NUM",
        "RRS412",
        "RRS443",
        "RRS490",
        "RRS555",
        "RRS670",
        "PAR",
        "KD490"
    ])
else:
    df_matchups = pd.read_csv(
        matchup_csv_file,
        delimiter=",",
        skiprows=0
    )

# Loop through each tag position and date
# go_to_next_day = False

for i, (lat, lon, date, tag, profile, ncfile, isdaytime, processed) in enumerate(zip(all_lat, all_lon, all_dates, all_ids, all_profilnum, all_ncfiles, all_isdaytime, all_processed)):
    
    # # Stop if we have at least 20 matchups (for testing)
    # if len(df_matchups) >= 20:
    #     break
    
    # Skip profiles that have already been processed
    if processed == True:
        print(f"{Fore.GREEN + Style.BRIGHT}Skipping tag {tag}, profile {profile}/{len(all_lat)}: lat={lat}, lon={lon}, date={date} (already processed){Style.RESET_ALL}")
        continue
    
    # Skip nighttime profiles
    if isdaytime == False:
        print(f"{Fore.RED + Style.BRIGHT}Skipping tag {tag}, profile {profile}/{len(all_lat)}: lat={lat}, lon={lon}, date={date} (nighttime profile){Style.RESET_ALL}")
        # Mark profile as processed in profile CSV
        df_profiles.at[i, "PROCESSED"] = True
        df_profiles.to_csv(profile_csv_file, index=False)
        continue
    
    print(f"{Fore.BLUE + Style.BRIGHT}Processing tag {tag}, profile {profile} ({i}/{len(all_lat)}): lat={lat}, lon={lon}, date={date}{Style.RESET_ALL}")

    # if go_to_next_day:
    #     if date == date_to_skip:
    #         print(f"Skipping date {date} as previously determined.")
    #         continue
    #     else:
    #         go_to_next_day = False
    
    # Extract Rrs from PACE AOP
    print(f"--> Extracting Rrs from PACE AOP")
    df_Rrs = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        par="Rrs",
        sat="PACE_AOP",
        selected_dates=[date],
        )
    
    # If valid Rrs data found, extract Kd from PACE IOP
    if len(df_Rrs)>0 and df_Rrs["sat_pixel_valid"].sum()>0:
        print(f"--> Extracting Kd from PACE IOP")
        df_Kd = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            wavelengths=[490],
            par="Kd",
            sat="PACE_IOP",
            selected_dates=[date],
            )
        
        print(f"--> Extracting PAR from PACE PAR")
        df_PAR = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            par="PAR",
            sat="PACE_PAR",
            selected_dates=[date],
            )
        
        # Merge Rrs, Kd, and PAR dataframes on sat_datetime
        df_satellite = pd.merge(
            df_Rrs,
            df_PAR,
            on=["sat_datetime", "sat_latitude", "sat_longitude", "sat_pixel_valid"],
            how="outer"
        )
        df_satellite = pd.merge(
            df_satellite,
            df_Kd,
            on=["sat_datetime", "sat_latitude", "sat_longitude", "sat_pixel_valid"],
            how="outer"
        )
        
        new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [ncfile],
                "TAG_ID": [tag],
                "PROFILE_NUM": [profile],
                "RRS412": [np.nanmean(df_satellite['sat_rrs413'].values)],
                "RRS443": [np.nanmean(df_satellite['sat_rrs442'].values)],
                "RRS490": [np.nanmean(df_satellite['sat_rrs490'].values)],
                "RRS555": [np.nanmean(df_satellite['sat_rrs555'].values)],
                "RRS670": [np.nanmean(df_satellite['sat_rrs670'].values)],
                "PAR": np.nanmean(df_satellite['sat_par'].values),
                "KD490": np.nanmean(df_satellite['sat_kd490'].values)
                
            })
        df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
        
        # Save intermediate results to CSV
        df_matchups.to_csv(matchup_csv_file, index=False)
        
        print(f"{Fore.GREEN}    Successfully extracted satellite data for date {date}.{Style.RESET_ALL}")
    else:
        print(f"{Fore.RED}    No valid satellite data for date {date}.{Style.RESET_ALL}")
    
    # Mark profile as processed in profile CSV
    df_profiles.at[i, "PROCESSED"] = True
    df_profiles.to_csv(profile_csv_file, index=False)
        
    # # If no valid data, skip to next day
    # if df_Rrs["sat_pixel_valid"].sum()==0:
    #     print(f"No valid satellite data for date {date}.")
    #     date_to_skip = date
    #     go_to_next_day = True


Skipping tag ct182-865F-24, profile 1/2317: lat=-49.5598, lon=70.3596, date=2025-01-24 (already processed)
Skipping tag ct182-865F-24, profile 2/2317: lat=-49.58, lon=70.3964, date=2025-01-24 (already processed)
Skipping tag ct182-865F-24, profile 3/2317: lat=-49.9602, lon=70.6291, date=2025-01-24 (already processed)
Skipping tag ct182-865F-24, profile 4/2317: lat=-50.0175, lon=70.6724, date=2025-01-24 (already processed)
Skipping tag ct182-865F-24, profile 5/2317: lat=-50.0387, lon=70.7835, date=2025-01-24 (already processed)
Skipping tag ct182-865F-24, profile 6/2317: lat=-50.4035, lon=70.9532, date=2025-01-25 (already processed)
Skipping tag ct182-865F-24, profile 7/2317: lat=-50.7647, lon=71.6812, date=2025-01-25 (already processed)
Skipping tag ct182-865F-24, profile 8/2317: lat=-51.1433, lon=72.1528, date=2025-01-26 (already processed)
Skipping tag ct182-865F-24, profile 9/2317: lat=-51.9319, lon=73.1725, date=2025-01-26 (already processed)
Skipping tag ct182-865F-24, profile 10/

/var/folders/t1/9ygzcs0j2ms2w3wgpmhfdhn1857szx/T/ipykernel_23481/1133343099.py:118: RuntimeWarning: Mean of empty slice
  "PAR": np.nanmean(df_satellite['sat_par'].values),


    Successfully extracted satellite data for date 2025-02-02.
Processing tag ct182-866F-24, profile 85 (744/2317): lat=-67.1479, lon=79.8755, date=2025-02-02
--> Extracting Rrs from PACE AOP
    Filtered to 4 Granules.
    Running Granule: 2025-02-02 07:15:15+00:00
    Running Granule: 2025-02-02 08:53:34+00:00
    Running Granule: 2025-02-02 10:26:53+00:00
    Running Granule: 2025-02-02 16:55:12+00:00
    No valid satellite data for date 2025-02-02.
Processing tag ct182-866F-24, profile 86 (745/2317): lat=-67.0605, lon=79.9486, date=2025-02-03
--> Extracting Rrs from PACE AOP
    Filtered to 4 Granules.
    Running Granule: 2025-02-03 07:49:58+00:00


Exception ignored in: <function File.close at 0x3322bbf60>
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/lib/python3.13/site-packages/h5netcdf/core.py", line 1844, in close
    self.flush()
  File "/opt/homebrew/anaconda3/lib/python3.13/site-packages/h5netcdf/core.py", line 1795, in flush
    if self._writable:
AttributeError: 'File' object has no attribute '_writable'


    Running Granule: 2025-02-03 09:28:17+00:00
    Running Granule: 2025-02-03 11:01:35+00:00
    Running Granule: 2025-02-03 15:51:36+00:00
--> Extracting Kd from PACE IOP
    Filtered to 4 Granules.
    Running Granule: 2025-02-03 07:49:58+00:00
    Running Granule: 2025-02-03 09:28:17+00:00
    Running Granule: 2025-02-03 11:01:35+00:00
    Running Granule: 2025-02-03 15:51:36+00:00
--> Extracting PAR from PACE PAR
    Filtered to 4 Granules.
    Running Granule: 2025-02-03 07:49:58+00:00
    Running Granule: 2025-02-03 09:28:17+00:00
    Running Granule: 2025-02-03 11:01:35+00:00
    Running Granule: 2025-02-03 15:51:36+00:00


...

    Successfully extracted satellite data for date 2025-03-05.
Skipping tag ct182-966F-24, profile 130/2317: lat=-67.2051, lon=80.2191, date=2025-03-05 (nighttime profile)
Processing tag ct182-966F-24, profile 131 (1644/2317): lat=-67.2164, lon=79.9914, date=2025-03-06
--> Extracting Rrs from PACE AOP
    Filtered to 3 Granules.
    Running Granule: 2025-03-06 08:03:14+00:00
    Running Granule: 2025-03-06 09:41:36+00:00
    Running Granule: 2025-03-06 11:19:58+00:00
    No valid satellite data for date 2025-03-06.
Processing tag ct182-966F-24, profile 132 (1645/2317): lat=-67.2467, lon=79.5453, date=2025-03-06
--> Extracting Rrs from PACE AOP
    Filtered to 3 Granules.
    Running Granule: 2025-03-06 08:03:14+00:00
    Running Granule: 2025-03-06 09:41:36+00:00
    Running Granule: 2025-03-06 11:19:58+00:00
    No valid satellite data for date 2025-03-06.
Processing tag ct182-966F-24, profile 133 (1646/2317): lat=-67.3001, lon=79.557, date=2025-03-06
--> Extracting Rrs from PACE AOP


/opt/homebrew/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/homebrew/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:216: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/homebrew/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


--> Extracting PAR from PACE PAR
    Filtered to 2 Granules.
    Running Granule: 2025-06-08 07:39:43+00:00
    Running Granule: 2025-06-08 09:18:04+00:00


/var/folders/t1/9ygzcs0j2ms2w3wgpmhfdhn1857szx/T/ipykernel_23481/1133343099.py:119: RuntimeWarning: Mean of empty slice
  "KD490": np.nanmean(df_satellite['sat_kd490'].values)


    Successfully extracted satellite data for date 2025-06-08.
Skipping tag ct182-969F-24, profile 345/2317: lat=-52.9878, lon=74.3667, date=2025-06-08 (nighttime profile)
Processing tag ct182-969F-24, profile 346 (2058/2317): lat=-52.8504, lon=74.3931, date=2025-06-09
--> Extracting Rrs from PACE AOP
    Filtered to 2 Granules.
    Running Granule: 2025-06-09 08:14:57+00:00
    Running Granule: 2025-06-09 09:53:19+00:00
    No valid satellite data for date 2025-06-09.
Skipping tag ct182-969F-24, profile 347/2317: lat=-52.7927, lon=74.294, date=2025-06-09 (nighttime profile)
Processing tag ct182-969F-24, profile 348 (2060/2317): lat=-52.9624, lon=74.2698, date=2025-06-10
--> Extracting Rrs from PACE AOP
    Filtered to 1 Granules.
    Running Granule: 2025-06-10 08:50:13+00:00
    No valid satellite data for date 2025-06-10.
Processing tag ct182-969F-24, profile 349 (2061/2317): lat=-52.887, lon=74.1751, date=2025-06-11
--> Extracting Rrs from PACE AOP
    Filtered to 2 Granules.
    R